# Early-Exit CNN → Quantised TFLite → C (Apache TVM)

**Author:** Ajay Kumar M, Vishnu PS

**Pipeline**
1. Build / load a backbone from the model registry
2. Fine-tune on CIFAR-10
3. Select early-exit placements (cost-constrained search)
4. Attach and jointly train exit heads (backbone frozen)
5. Quantise to INT8 TFLite
6. Compile to C via TVM AOT + microTVM runtime

**Environment.** Everything below runs in ONE environment: TF 2.12 / Keras 2.12
(`tf_keras`), which is what the pinned `apache-tvm==0.14.dev273` frontend expects.
Do not mix Keras 3 model construction with this runtime: HDF5 files written by
Keras 3 store `InputLayer(batch_shape=...)` and will not deserialise here.

Run cell 1, restart the runtime, then run everything from cell 2 onward.

## 0 · Dependencies (run once, then RESTART RUNTIME)

In [ ]:
!pip install -q tensorflow==2.12.0 keras==2.12.0 tflite==2.1.0 numpy==1.23.5
!pip install -q apache-tvm==0.14.dev273
# RESTART THE RUNTIME HERE. Dependency-resolver errors during install are expected
# and can be ignored provided the version check in the next cell passes.

In [ ]:
!pip install "https://files.pythonhosted.org/packages/9c/b2/adb56267ba6b6f05204407a7745e0c188e39589b7e33f490d4aa406d84e8/apache_tvm-0.14.dev273-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"

## 1 · Configuration

In [ ]:
import os, pathlib, itertools
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tvm, tvm.relay as relay
from tvm.relay.backend import Executor, Runtime
from tvm.micro import export_model_library_format

SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED)

# ---- Paths -------------------------------------------------------------
DRIVE_ROOT   = "/content/drive/MyDrive/"
CACHE_DIR    = f"{DRIVE_ROOT}/cached_datasets"
WEIGHTS_DIR  = f"{DRIVE_ROOT}/finetuned_weights"
CONVERTED_DIR= f"{DRIVE_ROOT}/converted_models"

# ---- Data --------------------------------------------------------------
IMAGE_SIZE    = (64, 64, 3)
BATCH_SIZE    = 128
NUM_CLASSES   = 10
CIFAR_CLASSES = ['airplane','automobile','bird','cat','deer',
                 'dog','frog','horse','ship','truck']

# ---- Backbone ----------------------------------------------------------
STEM_STRIDE   = 2

# ---- Training ----------------------------------------------------------
BASE_OPTIMIZER  = "sgd"
BASE_LR         = 0.1
LR_SCHEDULE     = "cosine"
LR_DECAY_AT     = (0.5, 0.75)
LR_DECAY_FACTOR = 0.1
EPOCHS_FINETUNE = 100
AUGMENT         = True

LEARNING_RATE   = 5e-4
EPOCHS_EE       = 10
DROPOUT_RATE    = 0.1
FINETUNE_FROM_SCRATCH = False      # False -> load weights from WEIGHTS_DIR

FREEZE_FINAL_HEAD = True

# ---- Early exit --------------------------------------------------------
AUTO_SELECT_EXITS = True          # False -> use exit_layers from the registry
N_EXITS_TARGET    = 3             # upper bound; search reduces N if infeasible
N_EXITS_MIN       = 1
EE_LOSS_WEIGHTS   = [0.2, 0.4, 0.7, 1.0]

# ---- Exit-placement cost model ----------------------------------------
ALPHA_ACC         = 0.25          # accelerator cost factor on BB core (1.0 = no accel)
MIN_FIRST_DEPTH   = 10.0
LAST_DEPTH_RANGE  = (55.0, 90.0)
TARGET_LAST_DEPTH = 70.0
LAMBDA_BAL        = 10.0          # weight on the segment-balance term
EXIT_HEAD_STRIDE  = 1

# ---- TVM ---------------------------------------------------------------
TVM_OPT_LEVEL    = 3
QUANTIZE_FOR_TVM = True

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
for d in (CACHE_DIR, WEIGHTS_DIR, CONVERTED_DIR):
    os.makedirs(d, exist_ok=True)

## 2 · Backbone definitions

In [ ]:
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, Activation,
                                     Add, Dense)
from tensorflow.keras.regularizers import l2

def _conv_bn(x, filters, k, strides, name, bn=True):
    x = Conv2D(filters, k, strides=strides, padding="same",
               kernel_initializer="he_normal", kernel_regularizer=l2(1e-4),
               name=name)(x)
    if bn:
        x = BatchNormalization(name=name + "_bn")(x)
    return x

def resnet_v1_cifar(input_shape=IMAGE_SIZE, n=1, num_filters=16, stem_stride=STEM_STRIDE):
    '''MLPerf Tiny ResNet-8 (v1) feature extractor, include_top=False.'''
    inp = Input(shape=input_shape, name="input")
    # 64x64 input -> 32x32 feature map
    x = _conv_bn(inp, num_filters, 3, stem_stride, "stem_conv")
    x = Activation("relu", name="stem_out")(x)

    for s in range(1, 4):
        f = num_filters * 2 ** (s - 1)
        for b in range(1, n + 1):
            stride = 2 if (s > 1 and b == 1) else 1
            tag = f"stack{s}_block{b}"
            y = _conv_bn(x, f, 3, stride, f"{tag}_conv1")
            y = Activation("relu", name=f"{tag}_relu1")(y)
            y = _conv_bn(y, f, 3, 1, f"{tag}_conv2")
            if stride != 1 or x.shape[-1] != f:
                x = _conv_bn(x, f, 1, stride, f"{tag}_proj", bn=False)
            x = Add(name=f"{tag}_add")([x, y])
            x = Activation("relu", name=f"{tag}_out")(x)
    return tf.keras.Model(inp, x, name=f"resnet{6 * n + 2}_v1")

## 3 · Model registry

In [ ]:
def _load_h5(path):
    return lambda: tf.keras.models.load_model(path)

MODEL_REGISTRY = {
    "resnet8": {
        "name": "ResNet8",
        "build_fn": lambda: resnet_v1_cifar(IMAGE_SIZE, 1, 16, STEM_STRIDE),
        "exit_layers": ["stack1_block1_out", "stack2_block1_out"],
        "head_offset": -1,
        "max_exits": 2,
        "tag": f"stem{STEM_STRIDE}",
    },

    "mobilenet_v1": {
        "name": "MobileNetV1",
        "build_fn": lambda: tf.keras.applications.MobileNet(
            input_shape=IMAGE_SIZE, alpha=0.25, weights="imagenet", include_top=False),
        "exit_layers": ["conv_pw_2_relu", "conv_pw_4_relu", "conv_pw_6_relu"],
        "head_offset": -5,
    },
    "mobilenet_v2": {
        "name": "MobileNetV2",
        "build_fn": lambda: tf.keras.applications.MobileNetV2(
            input_shape=IMAGE_SIZE, alpha=0.35, weights="imagenet", include_top=False),
        "exit_layers": ["block_3_expand_relu", "block_6_expand_relu", "block_10_expand_relu"],
        "head_offset": -1,
    },
    "efficientnet_b0": {
        "name": "EfficientNetB0",
        "build_fn": lambda: tf.keras.applications.EfficientNetB0(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False),
        "exit_layers": ["block2a_expand_activation", "block3a_expand_activation",
                        "block5a_expand_activation"],
        "head_offset": -1,
        "expects_0_255": True,
    },
    "resnet50": {
        "name": "ResNet50",
        "build_fn": lambda: tf.keras.applications.ResNet50(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False),
        "exit_layers": ["conv2_block3_out", "conv3_block4_out", "conv4_block6_out"],
        "head_offset": -1,
    },
    "vgg16": {
        "name": "VGG16",
        "build_fn": lambda: tf.keras.applications.VGG16(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False),
        "exit_layers": ["block2_pool", "block3_pool", "block4_pool"],
        "head_offset": -1,
    },
    "nasnet_mobile": {
        "name": "NASNetMobile",
        "build_fn": lambda: tf.keras.applications.NASNetMobile(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False),
        "exit_layers": ["normal_concat_3", "normal_concat_6", "normal_concat_9"],
        "head_offset": -1,
        "tvm_custom_pipeline": False,   # branchy graph hangs the fold-before-QNN pipeline
        "tvm_opt_level": 1,
    },
}

In [ ]:
# ---------------- CHOOSE MODEL ----------------
MODEL_KEY = "resnet8"
# ----------------------------------------------
MODEL_CFG  = MODEL_REGISTRY[MODEL_KEY]
OUTPUT_DIR = pathlib.Path(f"./output_{MODEL_CFG['name']}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_EXITS_TARGET = min(N_EXITS_TARGET, MODEL_CFG.get("max_exits", N_EXITS_TARGET))
print(f"Model: {MODEL_CFG['name']} | exits<= {N_EXITS_TARGET} | out: {OUTPUT_DIR}")

## 4 · Data

In [ ]:
cache_path = os.path.join(CACHE_DIR, f"cifar{NUM_CLASSES}_{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}.npz")

if os.path.exists(cache_path):
    d = np.load(cache_path)
    x_train_all, y_train_all = d["x_train"], d["y_train"]
    x_test_all,  y_test_all  = d["x_test"],  d["y_test"]
    print(f"Loaded cache {cache_path}")
else:
    (xtr, ytr), (xte, yte) = tf.keras.datasets.cifar10.load_data()
    x_train_all = tf.image.resize(xtr.astype("float32") / 255.0, IMAGE_SIZE[:2]).numpy()
    x_test_all  = tf.image.resize(xte.astype("float32") / 255.0, IMAGE_SIZE[:2]).numpy()
    y_train_all = tf.keras.utils.to_categorical(ytr.squeeze(), NUM_CLASSES)
    y_test_all  = tf.keras.utils.to_categorical(yte.squeeze(), NUM_CLASSES)
    np.savez_compressed(cache_path, x_train=x_train_all, y_train=y_train_all,
                        x_test=x_test_all, y_test=y_test_all)
    print(f"Cached -> {cache_path}")

idx = np.random.default_rng(SEED).permutation(len(x_train_all))
n_tr = int(0.8 * len(idx))
tr_idx, va_idx = idx[:n_tr], idx[n_tr:]

AUGMENT_FN = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", seed=SEED),
    tf.keras.layers.RandomTranslation(0.125, 0.125, fill_mode="reflect", seed=SEED),
], name="augment")

def _ds(x, y, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y)).batch(BATCH_SIZE)
    if augment:
        ds = ds.map(lambda a, b: (AUGMENT_FN(a, training=True), b),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = _ds(x_train_all[tr_idx], y_train_all[tr_idx], augment=AUGMENT)
val_ds   = _ds(x_train_all[va_idx], y_train_all[va_idx])
full_ds  = _ds(x_train_all, y_train_all)
test_images, test_labels = x_test_all, y_test_all

print(f"train {len(train_ds)} / val {len(val_ds)} / full {len(full_ds)} batches, "
      f"test {len(test_images)} images")

## 5 · Fine-tune the backbone

In [ ]:
pretrained = MODEL_CFG["build_fn"]()
backbone_model = tf.keras.Model(
    inputs=pretrained.inputs,
    outputs=pretrained.layers[MODEL_CFG["head_offset"]].output,
    name=f"{MODEL_CFG['name']}_backbone",
)
assert len(backbone_model.output_shape) == 4, (
    f"head_offset={MODEL_CFG['head_offset']} does not yield a 4-D feature map "
    f"(got {backbone_model.output_shape}); fix the registry entry.")

layers = [tf.keras.layers.InputLayer(input_shape=IMAGE_SIZE)]
if MODEL_CFG.get("expects_0_255"):
    layers.append(tf.keras.layers.Rescaling(255.0))
layers += [
    backbone_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(DROPOUT_RATE),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
]
base_model = tf.keras.Sequential(layers, name=f"{MODEL_CFG['name']}_base")
base_model.compile(
    optimizer=(tf.keras.optimizers.SGD(BASE_LR, momentum=0.9, nesterov=True)
               if BASE_OPTIMIZER == "sgd" else tf.keras.optimizers.Adam(BASE_LR)),
    loss="categorical_crossentropy", metrics=["accuracy"])
base_model.summary()

In [ ]:
RUN_TAG = f"{MODEL_CFG['name']}_cifar{NUM_CLASSES}_{IMAGE_SIZE[0]}" \
          + (f"_{MODEL_CFG['tag']}" if MODEL_CFG.get("tag") else "")
weights_path = os.path.join(WEIGHTS_DIR, f"{RUN_TAG}.weights.h5")

def lr_at(epoch):
    if LR_SCHEDULE == "cosine":
        return float(BASE_LR * 0.5 * (1 + np.cos(np.pi * epoch / EPOCHS_FINETUNE)))
    lr = BASE_LR
    for frac in LR_DECAY_AT:
        if epoch >= int(frac * EPOCHS_FINETUNE):
            lr *= LR_DECAY_FACTOR
    return lr

print(f"LR '{LR_SCHEDULE}' over {EPOCHS_FINETUNE} epochs: "
      f"start {lr_at(0):.4g} | mid {lr_at(EPOCHS_FINETUNE // 2):.4g} | "
      f"end {lr_at(EPOCHS_FINETUNE - 1):.4g}")
assert lr_at(EPOCHS_FINETUNE - 1) < 0.05 * BASE_LR, \
    "LR never decays over this budget. Check LR_SCHEDULE / EPOCHS_FINETUNE."

if FINETUNE_FROM_SCRATCH:
    history = base_model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE, verbose=2,
        callbacks=[tf.keras.callbacks.LearningRateScheduler(lr_at),
                   tf.keras.callbacks.ModelCheckpoint(
                       weights_path, monitor="val_accuracy", mode="max",
                       save_best_only=True, save_weights_only=True)])
    base_model.load_weights(weights_path)   # best epoch, not the last one
    print(f"Best weights -> {weights_path}")
else:
    base_model.load_weights(weights_path)
    history = None
    print(f"Loaded weights <- {weights_path}")

base_eval = base_model.evaluate(_ds(test_images, test_labels), verbose=0)
print(f"Single-exit test accuracy: {base_eval[1]:.4f}")

In [ ]:
def plot_history(hist, title="", save_path=None):
    if hist is None:
        print(f"No history for '{title}' - skipped.")
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, m in zip(axes, ["loss", "accuracy"]):
        train_key, val_key = m, f"val_{m}"
        if train_key not in hist.history:
            cands = [k for k in hist.history
                     if k.endswith(f"_{m}") and not k.startswith("val_")]
            if not cands:
                continue
            train_key, val_key = cands[-1], f"val_{cands[-1]}"
        ax.plot(hist.history[train_key], label="train")
        ax.plot(hist.history[val_key], label="val")
        ax.set(xlabel="Epoch", ylabel=m.title(), title=f"{title}{m.title()}")
        ax.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

plot_history(history, f"{MODEL_CFG['name']} base - ",
             OUTPUT_DIR / f"{MODEL_CFG['name']}_base_history.png")

In [ ]:
backbone = next(l for l in base_model.layers if isinstance(l, tf.keras.Model))
backbone.summary()

## 6 · Exit head

In [ ]:
def make_exit_head(features, name, exit_idx=1):
    '''Exit head. Deeper exits get one extra 3x3 conv.'''
    x = tf.keras.layers.Conv2D(32, 3, strides=EXIT_HEAD_STRIDE, padding="same",
                               activation="relu", name=f"conv1_{name}")(features)
    if exit_idx >= 2:
        x = tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu",
                                   name=f"conv2_{name}")(x)
    x = tf.keras.layers.GlobalAveragePooling2D(name=f"gap_{name}")(x)
    x = tf.keras.layers.Dense(128, activation="relu", name=f"dense_mid_1_{name}")(x)
    x = tf.keras.layers.Dense(64, activation="relu", name=f"dense_mid_2_{name}")(x)
    return tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name=name)(x)

def exit_head_macs(feat_shape, exit_idx=1):
    '''Closed-form MAC count for make_exit_head. Must mirror it exactly.'''
    h, w, c = feat_shape
    ho, wo = -(-h // EXIT_HEAD_STRIDE), -(-w // EXIT_HEAD_STRIDE)
    m  = ho * wo * 9 * c * 32                    # conv1
    ch = 32
    if exit_idx >= 2:
        m += ho * wo * 9 * 32 * 16               # conv2
        ch = 16
    m += ho * wo * ch                            # GAP
    m += ch * 128 + 128 * 64 + 64 * NUM_CLASSES  # dense
    return m

## 7 · Exit placement

**Cost model.** Only the BB core reaches the dot-product accelerator. Backbone
MACs that are accelerator-eligible (1x1 Conv2D, Dense) are scaled by
`ALPHA_ACC`; every other backbone MAC and *all* exit-head MACs are charged at
full software cost, because the EE core runs the heads without the accelerator.

**Feasibility constraint.** `head_cost[k] < segment_cost[k]` for every exit,
where segment k is the backbone work between exit k and exit k+1 (and, for the
last exit, from the exit to the end of the backbone). If no configuration
satisfies this at `N_EXITS_TARGET`, N is decremented down to `N_EXITS_MIN`.

In [ ]:
def _shape(s):
    return s[0] if isinstance(s, list) else s

def split_layer_macs(layer):
    '''-> (accelerator-eligible MACs, software-only MACs).'''
    out, inp = _shape(layer.output_shape), _shape(layer.input_shape)
    t = type(layer).__name__
    if t == "Conv2D":
        k = layer.kernel_size
        macs = k[0] * k[1] * inp[-1] * layer.filters * out[1] * out[2]
        return (macs, 0) if k == (1, 1) else (0, macs)
    if t == "DepthwiseConv2D":
        k = layer.kernel_size
        return 0, k[0] * k[1] * inp[-1] * layer.depth_multiplier * out[1] * out[2]
    if t == "Dense":
        return inp[-1] * layer.units, 0
    if t == "GlobalAveragePooling2D":
        return 0, inp[1] * inp[2] * inp[3]
    if t == "BatchNormalization":
        return 0, 4 * out[-1]
    return 0, 0

def backbone_cost(layer):
    a, s = split_layer_macs(layer)
    return ALPHA_ACC * a + s

In [ ]:
def find_exit_placement(backbone, n_target, n_min):
    cumulative = np.cumsum([backbone_cost(l) for l in backbone.layers])
    total_bb   = cumulative[-1]
    print(f"Backbone: {len(backbone.layers)} layers, {total_bb:,.0f} effective ops "
          f"(alpha_acc={ALPHA_ACC})")

    BOUNDARY_KEYWORDS = ["add", "_out", "_pool", "_concat", "pool_conv", "project/batchnorm"]
    candidates = []
    for i, layer in enumerate(backbone.layers):
        shape = _shape(layer.output_shape)
        if len(shape) != 4 or shape[1] is None:
            continue
        is_boundary = (any(k in layer.name.lower() for k in BOUNDARY_KEYWORDS)
                       or isinstance(layer, (tf.keras.layers.Concatenate,
                                             tf.keras.layers.MaxPooling2D,
                                             tf.keras.layers.AveragePooling2D)))
        candidates.append({"idx": i, "name": layer.name, "shape": shape,
                           "cum_cost": cumulative[i],
                           "depth_pct": cumulative[i] / total_bb * 100,
                           "is_boundary": is_boundary})

    dedup = {}
    for c in candidates:
        dedup[round(c["cum_cost"], 3)] = c
    candidates = sorted(dedup.values(), key=lambda c: c["idx"])

    boundary = [c for c in candidates if c["is_boundary"]]
    if len(boundary) >= 2 * n_target:
        candidates = boundary
    print(f"Candidate exit points: {len(candidates)}")
    for c in candidates:
        print(f"  [{c['idx']:3d}] {c['name']:<40s} shape={tuple(c['shape'][1:])} "
              f"depth={c['depth_pct']:5.1f}%{' *' if c['is_boundary'] else ''}")

    head_cost = {(c["idx"], e): exit_head_macs(tuple(c["shape"][1:]), e)
                 for c in candidates for e in range(1, n_target + 1)}

    def search(N):
        out = []
        for combo in itertools.combinations(range(len(candidates)), N):
            cs = [candidates[i] for i in combo]
            depths = [c["depth_pct"] for c in cs]
            if depths[0] < MIN_FIRST_DEPTH:
                continue
            if not (LAST_DEPTH_RANGE[0] <= depths[-1] <= LAST_DEPTH_RANGE[1]):
                continue
            segs = [cs[k + 1]["cum_cost"] - cs[k]["cum_cost"] for k in range(N - 1)]
            segs.append(total_bb - cs[-1]["cum_cost"])
            if any(s <= 0 for s in segs):
                continue
            hc = [head_cost[(cs[k]["idx"], k + 1)] for k in range(N)]
            if any(hc[k] >= segs[k] for k in range(N)):
                continue
            r = min(segs) / max(segs)
            out.append({"n_exits": N, "exits": [c["name"] for c in cs], "depths": depths,
                        "segments": segs, "head_costs": hc, "total_head_cost": sum(hc),
                        "margins": [segs[k] / hc[k] for k in range(N)],
                        "score": abs(depths[-1] - TARGET_LAST_DEPTH) - LAMBDA_BAL * r})
        out.sort(key=lambda x: (x["score"], x["total_head_cost"]))
        return out

    for N in range(n_target, n_min - 1, -1):
        found = search(N)
        print(f"\nN={N}: {len(found)} feasible configuration(s)")
        if not found:
            continue
        for rank, cfg in enumerate(found[:5], 1):
            print(f"  -- option {rank} --")
            for e in range(N):
                print(f"     exit {e+1}: {cfg['exits'][e]:<40s} ({cfg['depths'][e]:.1f}%)")
            print("     seg (eff): " + "  ".join(f"S{k+1}={s:,.0f}" for k, s in enumerate(cfg['segments'])))
            print("     head (raw):" + "  ".join(f"H{k+1}={h:,.0f}" for k, h in enumerate(cfg['head_costs'])))
            print("     margin S/H:" + "  ".join(f"{m:.2f}x" for m in cfg['margins']))
            print(f"     score: {cfg['score']:.3f}")
        return found[0]
    return None


if AUTO_SELECT_EXITS:
    chosen = find_exit_placement(backbone, N_EXITS_TARGET, N_EXITS_MIN)
    if chosen:
        MODEL_CFG["exit_layers"]  = chosen["exits"]
        MODEL_CFG["n_exits_used"] = chosen["n_exits"]
        print(f"\nSelected: {chosen['exits']}")
    else:
        print("\nNo feasible placement - falling back to registry exit_layers.")
        MODEL_CFG["n_exits_used"] = len(MODEL_CFG["exit_layers"])
else:
    MODEL_CFG["n_exits_used"] = len(MODEL_CFG["exit_layers"])
    print(f"Using registry exit_layers: {MODEL_CFG['exit_layers']}")

## 8 · Attach and train exit heads

The backbone is frozen; only the exit heads and the final classifier tail are
trainable.

In [ ]:
def build_multi_exit_model(base_model, cfg):
    bb = next(l for l in base_model.layers if isinstance(l, tf.keras.Model))
    bb.trainable = False

    feats = [bb.get_layer(n).output for n in cfg["exit_layers"]]
    bb_multi = tf.keras.Model(bb.input, feats + [bb.output], name="bb_multi")

    inp = tf.keras.Input(shape=IMAGE_SIZE, name="image")
    x = tf.keras.layers.Rescaling(255.0, name="rescale")(inp) if cfg.get("expects_0_255") else inp
    outs = bb_multi(x)

    exits = [make_exit_head(outs[i], f"early_exit_{i+1}", exit_idx=i + 1)
             for i in range(len(feats))]

    tail = outs[-1]
    bb_idx = base_model.layers.index(bb)
    for layer in base_model.layers[bb_idx + 1:]:
        if FREEZE_FINAL_HEAD:
            layer.trainable = False
        tail = layer(tail)
    exits.append(tail)
    if not FREEZE_FINAL_HEAD:
        print("WARNING: final head is trainable.")
    return tf.keras.Model(inp, exits, name=f"{cfg['name']}_multi_exit")

multi_exit_model = build_multi_exit_model(base_model, MODEL_CFG)
n_out = len(multi_exit_model.outputs)
print("outputs:", multi_exit_model.output_names)

In [ ]:
import hashlib

model_dir = pathlib.Path(WEIGHTS_DIR).parent / OUTPUT_DIR.name
model_dir.mkdir(parents=True, exist_ok=True)

bb_fp = hashlib.sha256(pathlib.Path(weights_path).read_bytes()).hexdigest()[:8]
ee_weights = model_dir / f"{RUN_TAG}_multi_exit_n{n_out - 1}.weights.h5"
print("backbone fingerprint:", bb_fp)

loss_weights = EE_LOSS_WEIGHTS[-n_out:]
multi_exit_model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=["categorical_crossentropy"] * n_out,
    loss_weights=loss_weights,
    metrics=["accuracy"],
)
print("loss weights:", dict(zip(multi_exit_model.output_names, loss_weights)))

_probe = _ds(test_images, test_labels)

def check_final_matches_baseline(tag):
    if not FREEZE_FINAL_HEAD:
        return
    ev = dict(zip(multi_exit_model.metrics_names,
                  multi_exit_model.evaluate(_probe, verbose=0)))
    acc = ev[f"{multi_exit_model.output_names[-1]}_accuracy"]
    assert abs(acc - base_eval[1]) < 1e-3, (
        f"[{tag}] final exit {acc:.4f} != single-exit baseline {base_eval[1]:.4f}. "
        "The backbone/tail were overwritten (usually by a stale exit checkpoint) "
        "or the freeze did not take. Re-run the fine-tune cell and delete "
        f"{ee_weights}.")
    print(f"[{tag}] final-exit == baseline ({acc:.4f})")

if ee_weights.exists():
    multi_exit_model.load_weights(str(ee_weights))
    base_model.load_weights(weights_path)
    ee_history = None
    print(f"Loaded <- {ee_weights}")
else:
    ee_history = multi_exit_model.fit(train_ds, validation_data=val_ds,
                                      epochs=EPOCHS_EE, verbose=2)
    multi_exit_model.save_weights(str(ee_weights))
    print(f"Saved -> {ee_weights}")

check_final_matches_baseline("post-load")

plot_history(ee_history, f"{MODEL_CFG['name']} multi-exit - ",
             OUTPUT_DIR / f"{MODEL_CFG['name']}_multi_exit_history.png")

In [ ]:
ev = multi_exit_model.evaluate(_ds(test_images, test_labels), verbose=0)
names = multi_exit_model.output_names
accs  = ev[-n_out:]
for nm, a in zip(names, accs):
    print(f"{nm:<20s} test acc {a:.4f}")
print(f"{'single-exit baseline':<20s} test acc {base_eval[1]:.4f}")

## 9 · INT8 quantisation

In [ ]:
def representative_dataset():
    for batch, _ in full_ds.take(100):        # 100 batches = 3200 images
        yield [batch]

conv = tf.lite.TFLiteConverter.from_keras_model(multi_exit_model)
fp32_path = OUTPUT_DIR / f"{MODEL_CFG['name']}_multi_exit_fp32.tflite"
fp32_path.write_bytes(conv.convert())
print(f"fp32 -> {fp32_path} ({fp32_path.stat().st_size:,} B)")

int8_path = None
if QUANTIZE_FOR_TVM:
    conv = tf.lite.TFLiteConverter.from_keras_model(multi_exit_model)
    conv.optimizations             = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset    = representative_dataset
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type      = tf.float32
    conv.inference_output_type     = tf.float32
    int8_path = OUTPUT_DIR / f"{MODEL_CFG['name']}_multi_exit_int8.tflite"
    int8_path.write_bytes(conv.convert())
    print(f"int8 -> {int8_path} ({int8_path.stat().st_size:,} B)")

tflite_to_compile_path = int8_path if QUANTIZE_FOR_TVM else fp32_path
print("selected for TVM:", tflite_to_compile_path)

## 10 · Compile to C

In [ ]:
import time

with open(tflite_to_compile_path, "rb") as f:
    tfl_bytes = f.read()
try:
    import tflite
    tfl_model = tflite.Model.GetRootAsModel(tfl_bytes, 0)
except AttributeError:
    import tflite.Model
    tfl_model = tflite.Model.Model.GetRootAsModel(tfl_bytes, 0)

mod, params = relay.frontend.from_tflite(tfl_model)
target   = tvm.target.Target("c")
runtime  = Runtime("crt")
executor = Executor("aot", {"unpacked-api": True, "interface-api": "c",
                            "workspace-byte-alignment": 8})
if params:
    mod["main"] = relay.build_module.bind_params_by_name(mod["main"], params)

USE_CUSTOM = MODEL_CFG.get("tvm_custom_pipeline", True)
OPT_LEVEL  = MODEL_CFG.get("tvm_opt_level", TVM_OPT_LEVEL)
t0 = time.time()

if USE_CUSTOM:
    print(f"custom fold-before-QNN pipeline, opt_level={OPT_LEVEL}")
    disabled = [] if QUANTIZE_FOR_TVM else ["qnn.Legalize", "qnn.Canonicalize"]
    with tvm.transform.PassContext(opt_level=OPT_LEVEL, disabled_pass=disabled,
                                   config={"tir.disable_vectorize": True}):
        mod = tvm.transform.Sequential([relay.transform.InferType(),
                                        relay.transform.FoldConstant(),
                                        relay.transform.SimplifyInference()])(mod)
        if QUANTIZE_FOR_TVM:
            mod = tvm.transform.Sequential([relay.qnn.transform.Legalize(),
                                            relay.qnn.transform.CanonicalizeOps()])(mod)
        compiled = relay.build(mod, target, runtime=runtime, executor=executor, params={})
else:
    print(f"standard pipeline, opt_level={OPT_LEVEL}")
    disabled = ["relay.AlterOpLayout"] + ([] if QUANTIZE_FOR_TVM
                                          else ["qnn.Legalize", "qnn.Canonicalize"])
    with tvm.transform.PassContext(opt_level=OPT_LEVEL, disabled_pass=disabled,
                                   config={"tir.disable_vectorize": True}):
        compiled = relay.build(mod, target, runtime=runtime, executor=executor, params={})

print(f"compiled in {time.time() - t0:.1f}s")
param_bytes = len(tvm.runtime.save_param_dict(params if params else compiled.get_params()))
print(f"params: {param_bytes:,} bytes")

tar = OUTPUT_DIR / (f"{MODEL_CFG['name']}_EE"
                    + ("_int8_aot.tar" if QUANTIZE_FOR_TVM else "_fp32_aot.tar"))
export_model_library_format(compiled, tar)
print(f"MLF -> {tar}")